In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Load both files ---
sensor_df = pd.read_csv('session_20260630_205845.csv')
phase_df = pd.read_csv('session_20260630_210910.csv')

# --- Convert to seconds relative to sensor session start ---
t0 = sensor_df['wallclock_ms'].iloc[0]
sensor_df['t_sec'] = (sensor_df['wallclock_ms'] - t0) / 1000
phase_df['t_sec'] = (phase_df['event_timestamp_ms'] - t0) / 1000

print(f"Sensor session duration: {sensor_df['t_sec'].iloc[-1]:.1f}s")
print(f"Phase log entries:\n{phase_df[['phase_name','t_sec']]}")

In [ ]:
# --- Plot all three sensors with phase boundaries overlaid ---
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(sensor_df['t_sec'], sensor_df['eda_conductance_us'])
axes[0].set_ylabel('EDA (µS)')
axes[0].set_title('Automated-Timer Calibration Session')

axes[1].plot(sensor_df['t_sec'], sensor_df['ppg_ir'], alpha=0.7)
axes[1].set_ylabel('PPG (IR)')

acc_mag = (sensor_df['acc_x']**2 + sensor_df['acc_y']**2 + sensor_df['acc_z']**2) ** 0.5
axes[2].plot(sensor_df['t_sec'], acc_mag)
axes[2].set_ylabel('Accel Magnitude')
axes[2].set_xlabel('Time (seconds)')

# Overlay phase boundaries -- these are now exact, not manually estimated
for ax in axes:
    for _, row in phase_df.iterrows():
        ax.axvline(row['t_sec'], color='gray', linestyle='--', alpha=0.6)
        if row['event_type'] == 'phase_start':
            ax.text(row['t_sec'], ax.get_ylim()[1], row['phase_name'],
                    rotation=90, fontsize=8, va='top')

plt.tight_layout()
plt.show()

In [ ]:
print(sensor_df['wallclock_ms'].diff().describe())